### Progress Report 2

In [6]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import librosa

# Sklean
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# TensorFlow
import tensorflow as tf
from tensorflow import keras
from keras import layers, optimizers, regularizers, callbacks, models
from pathlib import Path

In [2]:
# Json saving callback 

import json
import os

class SaveMetricsCallback(keras.callbacks.Callback):
  def __init__(self, filename="metrics.json"):
    super(SaveMetricsCallback, self).__init__()
    self.filename = filename
    self.history = []

    # Ensure the directory exists
    directory = os.path.dirname(self.filename)
    if directory and not os.path.exists(directory):
        os.makedirs(directory, exist_ok=True)

    # Load existing history if the file exists
    if os.path.exists(self.filename) and os.path.getsize(self.filename) > 0:
      try:
        with open(self.filename, 'r') as f:
          self.history = json.load(f)
      except json.JSONDecodeError:
        # Handle cases where the file is empty or corrupted JSON
        print(f"Warning: Could not decode JSON from {self.filename}. Starting with empty history.")
        self.history = []

  def on_epoch_end(self, logs=None):
    logs = logs or {}
    entry = {"epoch": len(self.history) + 1, **logs}
    self.history.append(entry)

  def on_train_end(self, logs=None):
    # Save the complete history to the file at the end of training
    with open(self.filename, 'w') as f:
      json.dump(self.history, f, indent=2)

In [3]:
# Input Shape
input_shape = (128, 130, 1)

# Log Directory Path 
log_dir = Path("/Users/wileyjones/Desktop/CS467/CNN_Music_Classifier/src/music_classifier/model/logs")

# Create a callback for early stopping

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    min_delta=0,
    restore_best_weights=True,
    verbose=1,
    mode='min')

# Learning Rate Scheduler

lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=10,
    min_lr=1e-6,
    verbose=1
)

# Better Preprocessing layer for audio files

preprocessing_layers = keras.Sequential([
    # 1. Random Translation: Slighly shifts the audio in time (X-axis)
    layers.RandomTranslation(height_factor=0, width_factor=0.1, fill_mode='constant'),

    # 2. Gaussian Noise: Forces the model to ignore background static
    layers.GaussianNoise(0.05)
])

In [4]:
# Spectogram Masking Function 

class SpecAugment(layers.Layer):
    """
    SpecAugmentation that removes parts of frequency bands to
    help fight overfitting.
    """
    def __init__(self, freq_mask_param=15, time_mask_param=15, **kwargs):
        super().__init__(**kwargs)
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param

    def call(self, x, training=None):
        if not training:
            return x   # no-op at inference time

        shape = tf.shape(x)
        batch, time_steps, freq_bins, channels = shape[0], shape[1], shape[2], shape[3]

        # ── Frequency mask ───────────────────────────────────────────────────
        f  = tf.random.uniform((), 0, self.freq_mask_param, dtype=tf.int32)
        f0 = tf.random.uniform((), 0, freq_bins - f,        dtype=tf.int32)
        freq_mask = tf.concat([
            tf.ones ([batch, time_steps, f0,             channels]),
            tf.zeros([batch, time_steps, f,              channels]),
            tf.ones ([batch, time_steps, freq_bins-f0-f, channels])
        ], axis=2)
        x = x * freq_mask

        # ── Time mask ────────────────────────────────────────────────────────
        t  = tf.random.uniform((), 0, self.time_mask_param, dtype=tf.int32)
        t0 = tf.random.uniform((), 0, time_steps - t,       dtype=tf.int32)
        time_mask = tf.concat([
            tf.ones ([batch, t0,              freq_bins, channels]),
            tf.zeros([batch, t,               freq_bins, channels]),
            tf.ones ([batch, time_steps-t0-t, freq_bins, channels])
        ], axis=1)
        x = x * time_mask

        return x

    def get_config(self):
        config = super().get_config()
        config.update({"freq_mask_param": self.freq_mask_param,
                        "time_mask_param": self.time_mask_param})
        return config

In [5]:
l2_strength = 0.0001 # Small L2

CNN_9_input = layers.Input(shape=input_shape, name="Mel_Spec")

# Preprocess Layers
x = preprocessing_layers(CNN_9_input)
x = SpecAugment(freq_mask_param=15, time_mask_param=15)(x)

# Layer 1: 32 filters
x = layers.Conv2D(32, (3, 3), activation="elu", padding="same",
                  kernel_regularizer=regularizers.l2(l2_strength))(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPool2D((2, 2))(x)
x = layers.Dropout(0.2)(x)

# Layer 2: 64 filters
x = layers.Conv2D(64, (3, 3), activation="elu", padding="same",
                  kernel_regularizer=regularizers.l2(l2_strength))(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPool2D((2, 2))(x)
x = layers.Dropout(0.2)(x)

# Layer 3: 128 filters (Expanded)
x = layers.Conv2D(128, (3, 3), activation="elu", padding="same",
                  kernel_regularizer=regularizers.l2(l2_strength))(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPool2D((2, 2))(x)
x = layers.Dropout(0.3)(x)

# Layer 4: 256 filters (New Layer)
x = layers.Conv2D(256, (3, 3), activation="elu", padding="same",
                  kernel_regularizer=regularizers.l2(l2_strength))(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPool2D((2, 2))(x)
x = layers.Dropout(0.3)(x)

x = layers.Flatten()(x)
x = layers.Dense(256, activation="elu")(x) # Increased from 128
x = layers.Dropout(0.5)(x)

# EXTRA DENSE LAYER
x = layers.Dense(128, activation="elu",
                 kernel_regularizer=regularizers.l2(l2_strength))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x) # A second layer of protection

# Final classification
CNN_9_output = layers.Dense(10, activation="softmax", name="Output")(x)

CNN_9 = keras.Model(CNN_9_input, CNN_9_output, name='CNN_9')

# Compile 
CNN_9.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy",
             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3)]
)

# Summary
CNN_9.summary()


Model: "CNN_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Mel_Spec (InputLayer)           │ (None, 128, 130, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 128, 130, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spec_augment (SpecAugment)      │ (None, 128, 130, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 130, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 130, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 65, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 65, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 65, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 65, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     4,194,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,89

 Total params: 4,619,018 (17.62 MB)

 Trainable params: 4,617,802 (17.62 MB)

 Non-trainable params: 1,216 (4.75 KB)

In [7]:
model = keras.models.load_model("../models//versions/best_model_v9_epoc_05_val_1.25.keras")

test = np.load("../dataset/test.npz")
X_test = test['X']
y_test = test['y']

loss, accuracy, _ = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

I0000 00:00:1778119927.087103 5286313 service.cc:153] XLA service 0x1657ad970 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778119927.087121 5286313 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1778119927.100571 5286313 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1778119927.217734 5314828 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.
I0000 00:00:1778119927.217780 5314828 rsqrt.cc:179] Falling back to 1 / sqrt(x) for f32 false


 2/62 ━━━━━━━━━━━━━━━━━━━━ 4s 72ms/step - accuracy: 0.1719 - loss: 1.7253 - sparse_top_k_categorical_accuracy: 0.8906  

I0000 00:00:1778119927.372732 5286313 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


62/62 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.5790 - loss: 1.3186 - sparse_top_k_categorical_accuracy: 0.8849
Test Loss: 1.3185755014419556
Test Accuracy: 0.5790281295776367
